# \[UPSIDE QA E2E TEST\] - Sub-Saharan African Bat Occurrence Records with Taxonomy, Locality, Coordinates, and Collection Metadata Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.hxvc-9mts/fair2.json`

The dataset describes occurrence records for bat species in sub-Saharan Africa, including taxonomy, locality, coordinates, source reference, and collection metadata.

> **Important note:** In Croissant datasets, record sets, fields, columns, and related entities should be referenced by their `@id` values. This notebook follows that principle throughout. It also includes logic to inspect the metadata dynamically because the provided package summary indicates that the top-level `recordSet` list may be empty or require runtime inspection from the Croissant document.

In [ ]:
# Ensure the required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the Croissant dataset with `mlcroissant`, then inspect the dataset metadata.

This section keeps metadata handling explicit and safe:
- `dataset.metadata` is treated as a single object
- we convert it to JSON only when needed for inspection
- we avoid subscripting `dataset.metadata` directly

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.hxvc-9mts/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object, then convert to JSON for inspection
metadata_obj = dataset.metadata
metadata_json = metadata_obj.to_json()

print('Dataset name:')
print(metadata_json.get('name', 'N/A'))
print('\nDescription:')
print(metadata_json.get('description', 'N/A'))
print('\nIdentifier:')
print(metadata_json.get('identifier', 'N/A'))
print('\nLicense:')
print(metadata_json.get('license', 'N/A'))
print('\nVersion:')
print(metadata_json.get('version', 'N/A'))

In [ ]:
# Optional: show a compact view of key metadata fields
summary_fields = [
    '@id', 'name', 'description', 'identifier', 'datePublished',
    'license', 'spatialCoverage', 'temporalCoverage', 'version'
]

summary = {field: metadata_json.get(field, None) for field in summary_fields}
pd.DataFrame(summary.items(), columns=['metadata_field', 'value'])

## 2. Data Overview
Review the available record sets and field definitions.

Because Croissant packages can vary in structure, we first inspect the metadata JSON to discover any available `recordSet` entries dynamically. All discovered record sets and fields are displayed using their `@id` values.

In [ ]:
# Discover record sets from the metadata JSON
record_sets_meta = metadata_json.get('recordSet', []) or []

print(f'Number of record sets declared in metadata: {len(record_sets_meta)}')

record_set_rows = []
for rs in record_sets_meta:
    record_set_rows.append({
        'record_set_@id': rs.get('@id'),
        'name': rs.get('name'),
        'description': rs.get('description'),
        'field_count': len(rs.get('field', []) or [])
    })

record_sets_df = pd.DataFrame(record_set_rows)
record_sets_df

In [ ]:
# Inspect fields for each record set using field @id values
field_rows = []

for rs in record_sets_meta:
    rs_id = rs.get('@id')
    for field in rs.get('field', []) or []:
        source_info = field.get('source', {}) if isinstance(field.get('source', {}), dict) else {}
        field_rows.append({
            'record_set_@id': rs_id,
            'field_@id': field.get('@id'),
            'field_name': field.get('name'),
            'description': field.get('description'),
            'dataType': field.get('dataType'),
            'source_fileObject_@id': source_info.get('fileObject'),
            'source_extract_column': source_info.get('extract', {}).get('column') if isinstance(source_info.get('extract', {}), dict) else None
        })

fields_df = pd.DataFrame(field_rows)
fields_df.head(20)

### Record preview
If at least one record set is available, the next cell previews a few records from the first discovered record set using its `@id`.

In [ ]:
record_set_ids = record_sets_df['record_set_@id'].dropna().tolist() if not record_sets_df.empty else []

if record_set_ids:
    preview_record_set_id = record_set_ids[0]
    print(f'Previewing records from record set @id: {preview_record_set_id}')
    for i, record in enumerate(dataset.records(record_set=preview_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print('No record sets were declared in metadata. This may indicate that the Croissant package exposes metadata without tabular record sets, or that record sets must be inspected externally.')

## 3. Data Extraction
Load records from each discovered record set into a pandas DataFrame.

This section uses variables for the record set IDs so the notebook remains reusable even if the dataset schema changes.

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded {len(dataframes[record_set_id])} rows from record set: {record_set_id}')
    except Exception as e:
        print(f'Could not load record set {record_set_id}: {e}')

print('\nAvailable DataFrames:')
print(list(dataframes.keys()))

In [ ]:
if dataframes:
    primary_record_set_id = list(dataframes.keys())[0]
    primary_df = dataframes[primary_record_set_id]
    print(f'Primary record set @id: {primary_record_set_id}')
    print('\nColumns:')
    print(primary_df.columns.tolist())
    display(primary_df.head())
else:
    primary_record_set_id = None
    primary_df = pd.DataFrame()
    print('No tabular record sets were loaded into DataFrames.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to the loaded data.

Because this dataset is described as containing occurrence records with geographic coordinates and collection metadata, we try to identify useful numeric and categorical fields dynamically.

Typical steps include:
- checking shape and missingness
- identifying numeric columns
- filtering rows on a numeric field
- normalizing a numeric field
- grouping by a categorical field

In [ ]:
if not primary_df.empty:
    print('Shape:', primary_df.shape)
    print('\nData types:')
    print(primary_df.dtypes)
    print('\nMissing values per column:')
    display(primary_df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))
else:
    print('EDA skipped because no primary DataFrame is available.')

In [ ]:
if not primary_df.empty:
    numeric_columns = primary_df.select_dtypes(include='number').columns.tolist()
    object_columns = primary_df.select_dtypes(include='object').columns.tolist()
    
    print('Numeric columns:')
    print(numeric_columns)
    print('\nObject/categorical-like columns:')
    print(object_columns)
else:
    numeric_columns = []
    object_columns = []

In [ ]:
# Choose a numeric field dynamically, preferring latitude/longitude/year-like columns when available
preferred_numeric_candidates = ['latitude', 'longitude', 'year', 'collection_year']

numeric_field = None
for candidate in preferred_numeric_candidates:
    if candidate in primary_df.columns and pd.api.types.is_numeric_dtype(primary_df[candidate]):
        numeric_field = candidate
        break

if numeric_field is None and numeric_columns:
    numeric_field = numeric_columns[0]

print('Selected numeric field:', numeric_field)

In [ ]:
if numeric_field is not None and not primary_df.empty:
    series = primary_df[numeric_field].dropna()
    if len(series) > 0:
        threshold = series.median()
        filtered_df = primary_df[primary_df[numeric_field] > threshold].copy()
        print(f'Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows')
        display(filtered_df.head())
        
        std_value = filtered_df[numeric_field].std()
        if pd.notna(std_value) and std_value != 0:
            filtered_df[f'{numeric_field}_normalized'] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
            )
        else:
            filtered_df[f'{numeric_field}_normalized'] = 0.0
        
        print(f'Normalized field: {numeric_field}_normalized')
        display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())
    else:
        filtered_df = primary_df.copy()
        print(f'Numeric field {numeric_field} has no non-null values.')
else:
    filtered_df = primary_df.copy()
    print('No numeric field available for filtering and normalization.')

In [ ]:
# Choose a grouping field dynamically, preferring common categorical fields from the dataset description
preferred_group_candidates = ['country', 'family', 'genus', 'species', 'checked', 'holotype']

group_field = None
for candidate in preferred_group_candidates:
    if candidate in filtered_df.columns:
        group_field = candidate
        break

if group_field is None and object_columns:
    group_field = object_columns[0]

print('Selected grouping field:', group_field)

if group_field is not None and numeric_field is not None and not filtered_df.empty:
    grouped_df = (
        filtered_df.groupby(group_field)[numeric_field]
        .agg(['count', 'mean', 'median', 'min', 'max'])
        .sort_values('count', ascending=False)
    )
    print(f'Grouped summary by {group_field}:')
    display(grouped_df.head(15))
else:
    grouped_df = pd.DataFrame()
    print('Grouping skipped because the necessary fields were not found.')

### Optional: inspect field IDs alongside loaded columns
The next cell helps connect Croissant field `@id` values with the columns present in the loaded DataFrame.

In [ ]:
if primary_record_set_id is not None and not fields_df.empty:
    display(fields_df[fields_df['record_set_@id'] == primary_record_set_id])
else:
    print('No field metadata available for the selected record set.')

## 5. Visualization
Visualize distributions and category counts from the dataset.

This section attempts a few robust visualizations that work when the relevant fields exist.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

if numeric_field is not None and not primary_df.empty and primary_df[numeric_field].dropna().shape[0] > 0:
    plt.figure(figsize=(8, 5))
    sns.histplot(primary_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
else:
    print('Skipping numeric distribution plot because no suitable numeric field was found.')

In [ ]:
if group_field is not None and not primary_df.empty and group_field in primary_df.columns:
    top_counts = primary_df[group_field].fillna('Missing').value_counts().head(15)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=top_counts.values, y=top_counts.index, orient='h')
    plt.title(f'Top categories in {group_field}')
    plt.xlabel('Count')
    plt.ylabel(group_field)
    plt.show()
else:
    print('Skipping categorical count plot because no suitable grouping field was found.')

In [ ]:
# Geographic scatter plot when latitude and longitude are available
lat_candidates = ['latitude', 'lat']
lon_candidates = ['longitude', 'lon', 'long']

lat_col = next((c for c in lat_candidates if c in primary_df.columns), None) if not primary_df.empty else None
lon_col = next((c for c in lon_candidates if c in primary_df.columns), None) if not primary_df.empty else None

if lat_col and lon_col:
    geo_df = primary_df[[lat_col, lon_col]].dropna()
    if not geo_df.empty:
        plt.figure(figsize=(8, 6))
        sns.scatterplot(data=geo_df, x=lon_col, y=lat_col, s=20, alpha=0.6)
        plt.title('Geographic distribution of occurrence records')
        plt.xlabel(lon_col)
        plt.ylabel(lat_col)
        plt.show()
    else:
        print('Latitude/longitude columns exist, but there are no complete coordinate pairs to plot.')
else:
    print('Skipping geographic scatter plot because latitude/longitude columns were not found.')

## 6. Conclusion
This notebook demonstrated how to:

- load a Croissant dataset from a URL using `mlcroissant`
- inspect dataset metadata safely through `dataset.metadata` as a single object
- discover record sets and fields using their `@id` values
- extract records into pandas DataFrames
- perform basic exploratory data analysis
- create simple visualizations for numeric, categorical, and geographic data when available

For this bat occurrence dataset, likely areas of further analysis include:
- taxonomic summaries by family, genus, and species
- geographic coverage by country or coordinate density
- temporal patterns in collection year
- missingness and quality checks for metadata such as museum number, holotype status, and checked status
- handling explicit uncertainty labels such as `cf.` taxa before downstream modeling

If the Croissant package is updated to expose additional record sets or richer field metadata, the same workflow can be reused with minimal changes because the notebook relies on dynamic discovery and `@id`-based references.